# LightStreamer PySpark Streaming Source - Basic Usage

This notebook demonstrates how to use the LightStreamer custom streaming source with PySpark.


## Setup

First, make sure you have the package installed and a LightStreamer server running.


In [ ]:
# Import required libraries
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json, current_timestamp
from pyspark.sql.types import StructType, StructField, StringType, DoubleType


## Create Spark Session


In [ ]:
spark = SparkSession.builder \
    .appName("LightStreamerExample") \
    .master("local[*]") \
    .config("spark.sql.streaming.checkpointLocation", "/tmp/checkpoint") \
    .getOrCreate()


## Connect to LightStreamer

Create a streaming DataFrame from LightStreamer. This example uses the demo stock list.


In [ ]:
# Configure LightStreamer connection
SERVER_URL = "http://localhost:8080"
ADAPTER_SET = "DEMO"
ITEMS = "item1,item2,item3,item4,item5"
FIELDS = "stock_name,last_price,time,pct_change,bid,ask,min,max"

# Create streaming DataFrame
df = spark.readStream \
    .format("lightstreamer") \
    .option("server_url", SERVER_URL) \
    .option("adapter_set", ADAPTER_SET) \
    .option("items", ITEMS) \
    .option("fields", FIELDS) \
    .option("mode", "MERGE") \
    .option("batch_size", "100") \
    .load()

# Show the schema
df.printSchema()


## Process the Stream

Apply transformations to the streaming data.


In [ ]:
# Cast numeric fields to appropriate types
processed_df = df \
    .withColumn("last_price", col("last_price").cast("double")) \
    .withColumn("pct_change", col("pct_change").cast("double")) \
    .withColumn("bid", col("bid").cast("double")) \
    .withColumn("ask", col("ask").cast("double")) \
    .withColumn("min", col("min").cast("double")) \
    .withColumn("max", col("max").cast("double")) \
    .withColumn("timestamp", current_timestamp())

# Filter for significant price changes
significant_changes = processed_df \
    .filter(col("pct_change") > 2.0)


## Write to Console

Display the streaming data in the console for testing.


In [ ]:
# Write to console
query = processed_df.writeStream \
    .outputMode("append") \
    .format("console") \
    .option("truncate", False) \
    .start()

# Wait for some data to arrive
import time
time.sleep(30)

# Stop the query
query.stop()
